# Lab Conversational Memory 

### Conversational Memory

Conversational memory is how chatbots can respond to our queries in a chat-like manner. It enables a coherent conversation, and without it, every query would be treated as an entirely independent input without considering past interactions.

The memory allows a _"agent"_ to remember previous interactions with the user. By default, agents are *stateless* — meaning each incoming query is processed independently of other interactions. The only thing that exists for a stateless agent is the current input, nothing else.

There are many applications where remembering previous interactions is very important, such as chatbots. Conversational memory allows us to do that.


In [ ]:
# !pip install -q langchain-cohere
# !pip install -q langchain-community

#

In [ ]:
COHERE_API_KEY=''

In [ ]:
from langchain import Cohere
from langchain_cohere import ChatCohere
from langchain.chains import LLMChain, ConversationChain
from langchain.chains.conversation.memory import (ConversationBufferMemory,
                                                  ConversationSummaryMemory,
                                                  ConversationBufferWindowMemory)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
llm = ChatCohere(cohere_api_key=COHERE_API_KEY,temperature=0,max_tokens=100)

In [ ]:
def get_AIresponse(chain, query):
    result = chain.run(query)
    return result

## What is memory?

**Definition**: Memory is an agent's capacity of remembering previous interactions with the user (think chatbots)


In [ ]:
conversation = ConversationChain(
    llm=llm,
)

In [ ]:
print(conversation.prompt.template)

So this chain's prompt is telling it to chat with the user and try to give truthful answers. If we look closely, there is a new component in the prompt **_history_**. This is where our memory will come into play.

Now that we've understood the basics of the chain we'll be using, we can get into memory.

## Memory types

### Memory type #1: ConversationBufferMemory

The `ConversationBufferMemory` does just what its name suggests: it keeps a buffer of the previous conversation as part of the context in the prompt.

**Key feature:** _the conversation buffer memory keeps the previous pieces of conversation completely unmodified, in their raw form._

In [ ]:
conversation_buf = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory()
)

We pass a user prompt the the `ConversationBufferMemory` 

In [ ]:
conversation_buf("Good morning AI!")

In [ ]:
response = get_AIresponse(
    conversation_buf,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge.Get me only specific 3 details"
)

print(response)

In [ ]:
response = get_AIresponse(
    conversation_buf,
    "I just want to analyze the different 3 possibilities. What can you think of?"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_buf,
    "Which data source types could be used to give context to the model?.Get me only 3 types"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_buf,
    "What is my aim again?"
)

print(response)

#

Our LLM with `ConversationBufferMemory` can clearly remember earlier interactions in the conversation. Let's take a closer look to how the LLM is saving our previous conversation. We can do this by accessing the `.buffer` attribute for the `.memory` in our chain.

In [ ]:
print(conversation_buf.memory.buffer)

### clear the memory to release the space

In [ ]:
conversation_buf.memory.clear()

In [ ]:
print(conversation_buf.memory.buffer)   # returns nothing

####  So every piece of our conversation has been explicitly recorded and sent to the LLM in the prompt.

##

### Memory type #2: ConversationSummaryMemory

 `ConversationSummaryMemory`.

Here we will keep a `summary` of our previous conversation snippets as our history. 

**Key feature:** _the conversation summary memory keeps the previous pieces of conversation in a summarized form, where the summarization is performed by an LLM._

In [ ]:
conversation_sum = ConversationChain(
    llm=llm,
    memory=ConversationSummaryMemory(llm=llm)
)

prompt used inside our conversation summary memory:

In [ ]:
print(conversation_sum.memory.prompt.template)

Each new interaction is summarized and appended to a running summary as the memory of our chain. 

In [ ]:
response = get_AIresponse(
    conversation_sum,
    "Good morning AI!"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_sum,
     "My interest here is to explore the potential of integrating Large Language Models with external knowledge.Get me only specific 3 details"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_sum,
     "I just want to analyze the different 3 possibilities. What can you think of?"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_sum,
    "Which data source types could be used to give context to the model?.Get me only 3 types"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_sum,
    "What is my aim again?"
)
print(response)

**Lets See whats in memory**

In [ ]:
print(conversation_sum.memory.buffer)

In [ ]:
conversation_buf.memory.clear()
print(conversation_buf.memory.buffer)

#

### Memory type #3: ConversationBufferWindowMemory

Another option is the `ConversationBufferWindowMemory` where we will be keeping a few of the last interactions in our memory but we will intentionally drop the oldest ones - short-term memory . We will control this window with the `k` parameter.

**Key feature:** _the conversation buffer window memory keeps the latest pieces of the conversation in raw form_

In [ ]:
conversation_bufw = ConversationChain(
    llm=llm,
    memory=ConversationBufferWindowMemory(k=1)
)

In [ ]:
response = get_AIresponse(
    conversation_bufw,
    "Good morning AI!"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_bufw,
    "My interest here is to explore the potential of integrating Large Language Models with external knowledge.Get me only specific 3 details"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_bufw,
    "I just want to analyze the different 3 possibilities. What can you think of?"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_bufw,
    "Which data source types could be used to give context to the model?.Get me only 3 types"
)
print(response)

In [ ]:
response = get_AIresponse(
    conversation_bufw,
    "What is my aim again?"
)
print(response)

Observe it effectively `forgot` what we talked about in the first interaction. Let's see what it 'remembers'. Given that we set `k` to be `1`, we would expect it remembers only the last interaction.

In [ ]:
bufw_history = conversation_bufw.memory.load_memory_variables(
    inputs=[]
)['history']

In [ ]:
print(bufw_history)

Only 1 immediate previous conversation preserved

`Advantage` we are shortening our conversation length when compared to buffer memory _without_ a window: